In [1]:
from dotenv import load_dotenv
import os
import vertexai
from vertexai.generative_models import GenerativeModel
load_dotenv()
project_id = "ai-signposting"
location="europe-west2"
vertexai.init(project=project_id, location=location)
model = GenerativeModel(model_name="gemini-1.5-flash-001") 
#get the database
mongodb_uri=os.environ.get("MONGO_URI")


In [2]:
from pymongo import MongoClient
client=MongoClient(mongodb_uri)
db=client["signposting_db"]

We can try this with the same input as with the previous file first:

In [4]:
def get_tagged_options(tag, location_choice, collection_name):
    collection=db[collection_name]
    formatted_tag="#"+tag
    options=collection.find({"Category tags": formatted_tag, "Local / National":location_choice}, projection={"_id":0})
    return list(options)
tag="benefits-advice"
location_choice="National"
collection_name="support_options"
options=get_tagged_options(tag, location_choice, collection_name)
for idx, option in enumerate(options):
    print(f"Option {idx+1}: {option}")

Option 1: {'Category tags': ['#debt-advice', '#budget-advice', '#benefits-advice'], 'Name': 'Step Change', 'Website': 'https://www.stepchange.org/', 'Phone - call': '0800 138 1111', 'Local / National': 'National', 'Postcode': '', 'Prefix': '', 'Short text description': 'Offers free comprehensive debt advice to support individuals gain financial control and stabilit', 'Logo-link': 'https://drive.google.com/uc?export=download&id=1kKCOELT1W2_vNdG-zj3PXIDmJCDGnxKC', 'Email': '', 'latitude': None, 'longitude': None}
Option 2: {'Category tags': ['#debt-advice', '#budget-advice', '#benefits-advice'], 'Name': 'Money Helper', 'Website': 'https://www.moneyhelper.org.uk/en', 'Phone - call': '', 'Local / National': 'National', 'Postcode': '', 'Prefix': '', 'Short text description': 'Offers free and impartial help with money and pensions, supported by the government', 'Logo-link': 'https://drive.google.com/uc?export=download&id=1IS934FbUi4zv8mE1GBvlSzgq70X54Pw_', 'Email': '', 'latitude': None, 'lon

We would probably extract just the relevant info prior to feeding each organization info object into the model:

In [7]:
def create_input_text(option):
    website=option["Website"]
    if option["Local / National"]=="National":
        location=option["Local / National"]
    else:
        location=option["Postcode"]
    description=option["Short text description"]
    name=option["Name"]
    result={"website": website, "location": location, "description": description, "name": name}
    return result
option_summaries=[create_input_text(option) for option in options]
print(option_summaries)

[{'website': 'https://www.stepchange.org/', 'location': 'National', 'description': 'Offers free comprehensive debt advice to support individuals gain financial control and stabilit', 'name': 'Step Change'}, {'website': 'https://www.moneyhelper.org.uk/en', 'location': 'National', 'description': 'Offers free and impartial help with money and pensions, supported by the government', 'name': 'Money Helper'}, {'website': 'https://capuk.org/', 'location': 'National', 'description': 'Offers free, expert debt help to support individual’s financial recovery and stability', 'name': 'Christians Against Poverty'}, {'website': 'https://www.uceplus.co.uk/about', 'location': 'National', 'description': 'Charity that aims to make essential financial information and support clear and accessible to the public', 'name': 'Universal Credit Essentials'}, {'website': 'https://www.moneywellness.com/debt-advice', 'location': 'National', 'description': 'Offers free financial guidance, promoting well-being through

Now we can make a prompt, first to just write a free form message with all relevant info (if sorting by postcode we would sort first, translation would be the last step most likely)

In [9]:
def describe_organization(option, model):
    prompt=f""" 
    This is a dictionary representing information on a support organization within the UK. 
    Using all of the information, write a description of the organization. 
    Make sure all details written in the dictionary are included. 
    Mention the name first. Include any additional details if you have knowledge of them. Keep your answer in the range of 3 sentences.
    Organization dictionary:
    {option}
    """
    response=model.generate_content(prompt) 
    print(response.text)
    print("\n")

In [11]:
import json
inputs=[json.dumps(option) for option in option_summaries]
model=GenerativeModel(model_name="gemini-1.5-flash-001")
for input_item in inputs:
    describe_organization(input_item, model)

Step Change is a national organization that provides free, comprehensive debt advice across the UK.  Their mission is to help individuals gain financial control and stability by offering support and guidance to navigate their debt situation.  Their website, https://www.stepchange.org/, is a valuable resource for anyone struggling with debt, providing information, tools, and access to their expert advisors. 



Money Helper is a national organization offering free and impartial advice on money and pensions.  They are supported by the UK government and can be reached through their website at https://www.moneyhelper.org.uk/en.  Money Helper is a valuable resource for individuals seeking guidance on managing their finances and planning for retirement. 



Christians Against Poverty (CAP) is a national organization based in the UK that provides free, expert debt help to individuals across the country.  Their mission is to support individuals' financial recovery and stability, empowering the

Since these are larger organizations, I think it is worthwhile to test with smaller local organizations as well, i.e foodbanks. To improve the prompt we can pass in the category that the user selected within the prompt as well.

In [14]:
user_selection="foodbank"
location_choice="Local" #to avoid complexity with postcodes for now we can just test all the local ones
foodbank_entries=get_tagged_options(user_selection, location_choice, collection_name)
print(foodbank_entries)

[{'Category tags': ['#foodbank'], 'Name': 'Bodmin foodbank', 'Website': 'https://www.wadebridgefoodbank.org/', 'Phone - call': '01208 815374', 'Local / National': 'Local', 'Postcode': 'PL31 2NS', 'Prefix': 'PL31', 'Short text description': 'Offers emergency food assistance to individuals and families in Bodmin facing food or financial difficulties', 'Logo-link': 'https://drive.google.com/uc?export=download&id=1tE2nOYNeBjLcQqGGk5wZxhXYf5BqVDzF', 'Email': '', 'latitude': 50.473578, 'longitude': -4.723725}, {'Category tags': ['#foodbank'], 'Name': 'Bude foodbank', 'Website': 'https://budefoodbank.org.uk/', 'Phone - call': '01288 356635', 'Local / National': 'Local', 'Postcode': 'EX23 8AR', 'Prefix': 'EX23', 'Short text description': 'Offers emergency food assistance to individuals and families in Bude facing food or financial difficulties', 'Logo-link': 'https://drive.google.com/uc?export=download&id=1jl7zePdof_Da-4krUlzn9D9T8PK4YuU3', 'Email': 'janelibretto@capuk.org', 'latitude': 50.827

In [15]:
foodbanks=[create_input_text(foodbank) for foodbank in foodbank_entries]
print(foodbanks)

[{'website': 'https://www.wadebridgefoodbank.org/', 'location': 'PL31 2NS', 'description': 'Offers emergency food assistance to individuals and families in Bodmin facing food or financial difficulties', 'name': 'Bodmin foodbank'}, {'website': 'https://budefoodbank.org.uk/', 'location': 'EX23 8AR', 'description': 'Offers emergency food assistance to individuals and families in Bude facing food or financial difficulties', 'name': 'Bude foodbank'}, {'website': 'https://www.wadebridgefoodbank.org/', 'location': 'PL32 9PB', 'description': 'Offers emergency food assistance to individuals and families in Camelford facing food or financial difficulties', 'name': 'Camelford foodbank'}, {'website': 'https://callington.foodbank.org.uk/', 'location': 'PL17 7BS', 'description': 'Offers emergency food assistance to individuals and families in Callington facing food or financial difficulties', 'name': 'Callington foodbank'}, {'website': 'https://launceston.foodbank.org.uk/about/launceston-foodbank-ch

In [18]:
inputs=[json.dumps(foodbank) for foodbank in foodbanks]
def describe_organization(option, model, category):
    prompt=f""" 
    This is a dictionary representing information on a support organization within the UK. 
    The organization has been categorized. The category is {category}.
    Using all of the information, write a description of the organization. 
    Make sure all details written in the dictionary are included, including a link to the website. 
    Mention the name first. Include any additional details if you have knowledge of them. Keep your answer in the range of 3 sentences.
    Organization dictionary:
    {option}
    """
    response=model.generate_content(prompt) 
    print(response.text)
    print("\n")

In [19]:
for input_item in inputs:
    describe_organization(input_item, model, user_selection)

Bodmin Foodbank, located in PL31 2NS, provides emergency food assistance to individuals and families in Bodmin who are experiencing food or financial difficulties. The organization aims to alleviate hunger and support those in need by offering food packages and other essential supplies. You can find more information and ways to get involved on their website: https://www.wadebridgefoodbank.org/. 



Bude foodbank is a vital resource for those in the EX23 8AR area experiencing food insecurity. They provide emergency food assistance to individuals and families facing difficult financial circumstances. You can find more information and ways to get involved on their website at https://budefoodbank.org.uk/.  



Camelford foodbank is a vital resource for individuals and families in Camelford, Cornwall (PL32 9PB) who are experiencing food or financial hardship. They provide emergency food assistance to help alleviate immediate needs. You can find out more about their services and how to acces

Since some of the responses are duplicate, I think this is where we could enrich it via a scraped home page. I'll hardcode this to test for now.

In [33]:
from selenium import webdriver
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.chrome.options import Options 
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
import time
def create_driver():
    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--remote-debugging-port=9222")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")
    driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=chrome_options)
    return driver

def scrape(url, wait_time=45):
        
        driver = create_driver()
        try:
            driver.get(url)
            time.sleep(2) 
            content = driver.page_source
            content_title = driver.title

        except TimeoutException:
            print(f"TimeoutException: The page at {url} did not load within {wait_time} seconds.")
            content, content_title = None, None
        except WebDriverException as e:
            print(f"WebDriverException: {e}")
            content, content_title = None, None
        finally:
            driver.quit()
        return content, content_title
    

In [34]:
url="https://callington.foodbank.org.uk/"
content=scrape(url)[0]

The input the LLM recieves:

In [36]:
from bs4 import BeautifulSoup as soup
html=soup(content, "html.parser")
page_text=html.get_text(separator=' ', strip=True)
print(page_text)

Callington Foodbank | Helping Local People in Crisis Your choice regarding cookies on this site We use cookies to optimise site functionality and give you the best possible experience. Accept Reject Cookie Preferences Skip to content We are part of a nationwide network of foodbanks, supported by the Trussell Trust. Find out how to get emergency food and other support. Make a food donation or find out about volunteering. Find out more Seeded by The Trussell Trust Menu Donate Home About Get Help Locations Give Help News Contact Us Donate Mental Health Support Privacy & Cookies Complaints Policy Callington Foodbank Helping local people in crisis Learn more 11506.87 Kg's Amount of food given out last year Open 2pm-4pm every Monday, Wednesday & Friday (Closed on Bank Holidays) with our warm space open from 10am 1089 people supported within Callington & surrounding areas in the last year (a 33% increase from the previous year) Get Help Find out how to get help from our foodbank. Click here >

Now we can use a prompt that will help the LLM using the extracted text:

In [42]:
def describe_organization_with_context(option, homepage_text, model, category):
    prompt=f""" 
    This is a dictionary representing information on a support organization within the UK. 
    The organization has been categorized. The category is {category}.
    Organization dictionary:
    {option}
    Acting as a helpful guide, using all of the information, write a focused, conscise and informative description of the organization.
    Your description must follow guidelines: 
    1. Make sure all details written in the dictionary are included, including a link to the website. 
    2. Use this text from the website homepage to extract any additional details and include them in the response.
    Homepage text:
    {context}
    3.Use only the context and the dictionary for your response. 
    4. Do not make anything up.
    5. Mention the name first.  
    """
    response=model.generate_content(prompt) 
    print(response.text)
    print("\n")

In [43]:
def get_page_content(url):
    content=scrape(url)[0]
    if content is not None:
        html=soup(content, "html.parser")
        page_text=html.get_text(separator=' ', strip=True)
    else:
        page_text=""
    return page_text

This loop would execute after a user sends a message, so I thought to time it as well to check

In [45]:
start_time = time.time()

# Loop through foodbanks and perform the operations
for foodbank in foodbanks[:5]: #I would slice off 5 of the closest options or just a random 5 (if national)
    url = foodbank["website"]
    context = get_page_content(url)
    describe_organization_with_context(foodbank, context, model, user_selection)


end_time = time.time()


elapsed_time = end_time - start_time

print(f"Time for scraping and full response: {elapsed_time:.2f} seconds")

Bodmin foodbank is a food bank located in Bodmin, Cornwall, UK, with an address of PL31 2NS. It provides emergency food assistance to individuals and families facing food or financial difficulties. You can find more information on their website: https://www.wadebridgefoodbank.org/.  



Bude Food Bank is a food bank located in Bude, Cornwall (EX23 8AR) that provides emergency food assistance to individuals and families facing food or financial difficulties. They offer a variety of services including food parcels, financial support, and emotional support. They are open Monday through Friday from 9:30 am to 1:00 pm (excluding Wednesday). You can contact them at 01288 356635 or 07545 959725. To learn more about Bude Food Bank, visit their website at https://budefoodbank.org.uk/. 



Camelford foodbank is a food bank located in PL32 9PB and offers emergency food assistance to individuals and families in Camelford facing food or financial difficulties. You can learn more about this organiza